# Perfil de colunas e qualidade

Este notebook prepara o profiling das bases DATASUS antes da definicao do schema analitico final.

## Pergunta operacional

Quais colunas, tipos, nulos, valores ignorados e variacoes por ano precisam ser considerados antes do schema final?

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from src.etl.datasus_sources import parse_years, resolve_datasus_sources
from src.etl.profile_datasus import profile_pair

ROOT = Path.cwd()
if not (ROOT / "README.md").exists():
    ROOT = ROOT.parent.parent

ANOS = parse_years(2024, None)
GERAR_PERFIL = False
SAIDA_PERFIL = ROOT / "data/profiles/datasus_column_profile.csv"

## Arquivos disponiveis

In [ ]:
pares, ausentes = resolve_datasus_sources(ANOS)

pd.DataFrame(
    [
        {
            "ano": par.year,
            "sinan": str(par.sinan.relative_to(ROOT)),
            "sinasc": str(par.sinasc.relative_to(ROOT)),
        }
        for par in pares
    ]
)

In [ ]:
ausentes

## Geracao do perfil

Por padrao, a geracao fica desativada para evitar leitura pesada acidental. Defina `GERAR_PERFIL = True` para criar o CSV em `data/profiles/`.

In [ ]:
if GERAR_PERFIL:
    linhas = []
    for par in pares:
        linhas.extend(profile_pair(par))

    perfil = pd.DataFrame(linhas)
    SAIDA_PERFIL.parent.mkdir(parents=True, exist_ok=True)
    perfil.to_csv(SAIDA_PERFIL, index=False, encoding="utf-8")
    display(perfil.head(20))
    print(f"Perfil salvo em: {SAIDA_PERFIL}")
else:
    print("Profiling nao executado. Defina GERAR_PERFIL = True para gerar o CSV.")

## Variaveis criticas para decisao de schema

- raca/cor materna
- pre-natal
- escolaridade
- idade materna
- momento do diagnostico
- tratamento materno
- municipio de residencia